In [1]:
!pip install transformers torch accelerate -q
!pip install "datasets==2.19.0" -q

In [2]:
from datasets import load_dataset

dataset = load_dataset("facebook/empathetic_dialogues")
print(dataset)
print(dataset['train'][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for facebook/empathetic_dialogues contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/facebook/empathetic_dialogues
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major relea

Generating train split:   0%|          | 0/76673 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/12030 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10943 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags'],
        num_rows: 76673
    })
    validation: Dataset({
        features: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags'],
        num_rows: 12030
    })
    test: Dataset({
        features: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags'],
        num_rows: 10943
    })
})
{'conv_id': 'hit:0_conv:1', 'utterance_idx': 1, 'context': 'sentimental', 'prompt': 'I remember going to the fireworks with my best friend. There was a lot of people_comma_ but it only felt like us in the world.', 'speaker_idx': 1, 'utterance': 'I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people_comma_ we felt like the only people in the world.', 'selfeval': '5|5|5_2|2|

Loading base model and token

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# GPT2-based models don't have a pad token by default
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Formating the data for training

In [5]:
def format_example(example):
    context = example['prompt']
    response = example['utterance']
    text = f"Context: {context}\nResponse: {response}{tokenizer.eos_token}"
    return {"text": text}

formatted_dataset = dataset['train'].map(format_example)
print(formatted_dataset[0]['text'])

Map:   0%|          | 0/76673 [00:00<?, ? examples/s]

Context: I remember going to the fireworks with my best friend. There was a lot of people_comma_ but it only felt like us in the world.
Response: I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people_comma_ we felt like the only people in the world.<|endoftext|>


In [8]:
# Use a smaller subset to keep training
small_dataset = formatted_dataset.shuffle(seed=42).select(range(5000))

Tokeninzing the txt(txt-ID)

In [9]:
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

tokenized_dataset = small_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Language Modeling

In [11]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # GPT-2 style models predict the next word, not masked words
)

Speeds up the training

In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./mental-health-chatbot",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=True,
    report_to="none"
)

Fine tune

In [16]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

Step,Training Loss
50,2.373098
100,2.337923
150,2.352210
200,2.370784
250,2.374965
300,2.373543
350,2.373970
400,2.370987
450,2.417270
500,2.403127


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1875, training_loss=2.281255891927083, metrics={'train_runtime': 225.7448, 'train_samples_per_second': 66.447, 'train_steps_per_second': 8.306, 'total_flos': 489931407360000.0, 'train_loss': 2.281255891927083, 'epoch': 3.0})

Saving fine-tune model

In [17]:
model.save_pretrained("./mental-health-chatbot-final")
tokenizer.save_pretrained("./mental-health-chatbot-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./mental-health-chatbot-final/tokenizer_config.json',
 './mental-health-chatbot-final/tokenizer.json')

Building CLI

In [18]:
from transformers import pipeline

chatbot = pipeline(
    "text-generation",
    model="./mental-health-chatbot-final",
    tokenizer="./mental-health-chatbot-final"
)

def chat():
    print("Mental Health Support Chatbot (type 'quit' to exit)\n")
    while True:
        user_input = input("You: ")
        if user_input.lower() == "quit":
            break
        prompt = f"Context: {user_input}\nResponse:"
        result = chatbot(
            prompt,
            max_new_tokens=60,
            num_return_sequences=1,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            temperature=0.8
        )
        response = result[0]['generated_text'].split("Response:")[-1].strip()
        print(f"Bot: {response}\n")

chat()

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Mental Health Support Chatbot (type 'quit' to exit)

You: Ive sore thorat


[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'num_return_sequences', 'pad_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanu

Bot: Ive been doing well for my age haha. I am getting older and I dont like to break my bones :( :(  I dont like to cross the aisle :( LOL   I cant believe the weight in my body.  Like when I had to cross the aisle_comma_ I feel

You: quit
